# Capstone 5 — Autonomous Research Analyst

You will build a **LangGraph multi-agent system** that, given a research question, produces a cited report:
```
Planner → Researcher (web + RAG) → Critic → Writer
```
With: human-in-the-loop on critic rejection, full traces, a budget cap, and a graded eval suite.

## Phase 0 — Concept Recall

1. Why split planning from execution into separate agents?
2. What does the **critic** prevent that a single LLM would miss?
3. Name three failure modes of multi-agent loops and the guardrail for each.

*(5 pts each.)*

In [ ]:
# Phase 1 — Setup
# pip install langgraph langchain-openai duckduckgo-search langfuse

from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
import operator, json, time

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class State(TypedDict):
    question: str
    plan: List[str]
    findings: Annotated[List[dict], operator.add]
    critique: str
    report: str
    cost_usd: Annotated[float, operator.add]
    steps: Annotated[int, operator.add]


In [ ]:
# Phase 2 — Nodes (Guided Build)

def planner(state: State) -> dict:
    # TODO 1: ask the LLM to break the question into 3-6 sub-tasks.
    # Return {"plan": [...], "steps": 1, "cost_usd": <est>}
    raise NotImplementedError

def researcher(state: State) -> dict:
    # TODO 2: for each plan item, run a web search OR a RAG retrieval.
    # Return {"findings": [{"task": ..., "sources": [...], "summary": ...}], "steps": ..., "cost_usd": ...}
    raise NotImplementedError

def critic(state: State) -> dict:
    # TODO 3: ask LLM to verify findings cover every plan item AND each claim has a source.
    # Return {"critique": "accept" | reasoning, "steps": 1, "cost_usd": ...}
    raise NotImplementedError

def writer(state: State) -> dict:
    # TODO 4: compose the final report with inline [n] citations + a sources list.
    raise NotImplementedError


In [ ]:
# Phase 3 — Graph wiring with conditional edges + budget cap

BUDGET_USD = 0.50
MAX_STEPS  = 20

def route_after_critic(state: State) -> str:
    if state["cost_usd"] >= BUDGET_USD or state["steps"] >= MAX_STEPS:
        return "writer"   # bail out, write what we have
    if state["critique"].startswith("accept"):
        return "writer"
    return "researcher"   # critic rejected -> loop

g = StateGraph(State)
g.add_node("planner", planner)
g.add_node("researcher", researcher)
g.add_node("critic", critic)
g.add_node("writer", writer)
g.set_entry_point("planner")
g.add_edge("planner", "researcher")
g.add_edge("researcher", "critic")
g.add_conditional_edges("critic", route_after_critic, {"researcher": "researcher", "writer": "writer"})
g.add_edge("writer", END)

app = g.compile(checkpointer=MemorySaver())


In [ ]:
# Phase 4 — Human-in-the-loop on rejection

# Use the checkpointer + interrupt to pause when the critic rejects too many times.
# from langgraph.types import interrupt
# In `critic`, after N rejections, call `interrupt("critic blocked – override?")`.
# Resume with: app.invoke(None, config={"configurable": {"thread_id": tid}})

# TODO 5: implement the interrupt + a CLI prompt for human override.


In [ ]:
# Phase 5 — Eval suite

EVAL = [
    # {"q": "...", "must_mention": ["factA", "factB"], "must_cite_n": 3},
    # TODO 6: 15+ research questions with hand-graded gold criteria.
]

def grade(report: str, item: dict) -> dict:
    return {
        "covers_facts":   sum(f.lower() in report.lower() for f in item["must_mention"]) / len(item["must_mention"]),
        "citation_count": report.count("["),
        "meets_cite_min": report.count("[") >= item["must_cite_n"],
    }

def run_eval():
    rows = []
    for item in EVAL:
        out = app.invoke({"question": item["q"], "plan": [], "findings": [], "critique": "", "report": "", "cost_usd": 0.0, "steps": 0},
                          config={"configurable": {"thread_id": item["q"][:30]}})
        rows.append({**grade(out["report"], item), "cost": out["cost_usd"], "steps": out["steps"]})
    return rows


In [ ]:
# Phase 6 — Observability (required)

# Wire Langfuse OR write JSONL traces. Every node entry/exit + every tool call.
import logging, uuid
trace_log = logging.getLogger("agent.trace")

def trace(node, state, t0):
    trace_log.info(json.dumps({
        "trace_id": state.get("trace_id"),
        "node": node,
        "latency_ms": int((time.time() - t0) * 1000),
        "cost_usd": state.get("cost_usd", 0.0),
        "steps": state.get("steps", 0),
    }))

# TODO 7: wrap each node so trace() runs at exit; add a span for every tool call inside researcher.


## Phase 7 — Submission checklist (rubric / 100)

- [ ] **25 pts** — Task success rate ≥ 0.7 on the 15-item eval (covers ≥ 70% of `must_mention` facts).
- [ ] **15 pts** — Plan quality (LLM-judge rubric): rated ≥ 4/5 average.
- [ ] **15 pts** — Tool-use efficiency: median run ≤ 8 steps and ≤ $0.20.
- [ ] **15 pts** — Citation accuracy: every `[n]` resolves to a real source.
- [ ] **10 pts** — Failure-mode handling: a tool error inside `researcher` does NOT crash the graph.
- [ ] **10 pts** — Observability: full per-run trace exported (Langfuse / JSONL).
- [ ] **10 pts** — Guardrails: budget cap fires correctly; HITL pause demonstrated once on video.

### What you will have *learned by doing*
- That a graph beats a chain the moment you need to loop or branch.
- That a critic node, even a cheap one, eliminates a class of dumb failures.
- That an agent without observability is a liability, not a product.